# 04 — Fine-tune PEGASUS-PubMed (Abstractive) on Colab GPU

Uses Hugging Face `Seq2SeqTrainer` to fine-tune `google/pegasus-pubmed` on our cleaned PubMed subset. Falls back to `sshleifer/distilbart-cnn-12-6` if PEGASUS doesn't fit on the available GPU.

In [ ]:
# Setup: clone repo + mount Drive + install deps. Needs GPU.
REPO_URL = 'https://github.com/Captain-Uchiha/scientific-paper-summarizer.git'
!rm -rf /content/scientific-summarizer
!git clone -q {REPO_URL} /content/scientific-summarizer
import sys, os
sys.path.insert(0, '/content/scientific-summarizer')
os.chdir('/content/scientific-summarizer')

!pip -q install 'transformers>=4.40' datasets accelerate sentencepiece rouge-score evaluate

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
PROJECT_DIR = '/content/drive/MyDrive/scientific-summarizer-data'
!nvidia-smi

In [ ]:
# Pick a model. If GPU runs out of memory on pegasus-pubmed, switch to distilbart.
import sys, os
if '/content/scientific-summarizer' not in sys.path:
    sys.path.insert(0, '/content/scientific-summarizer')
if os.path.isdir('/content/scientific-summarizer'):
    os.chdir('/content/scientific-summarizer')
MODEL_NAME = 'google/pegasus-pubmed'
# MODEL_NAME = 'sshleifer/distilbart-cnn-12-6'

MAX_INPUT = 1024
MAX_TARGET = 256
PROC = f'{PROJECT_DIR}/dataset/processed'
OUT_DIR = f'{PROJECT_DIR}/models/{MODEL_NAME.split("/")[-1]}-ft'
print('Saving to', OUT_DIR)

In [ ]:
from datasets import load_dataset

data_files = {
    'train': f'{PROC}/train.jsonl',
    'validation': f'{PROC}/val.jsonl',
    'test': f'{PROC}/test.jsonl',
}
ds = load_dataset('json', data_files=data_files)
print(ds)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(ex):
    inp = tok(ex['input_text'], max_length=MAX_INPUT, truncation=True)
    with tok.as_target_tokenizer():
        tgt = tok(ex['target_text'], max_length=MAX_TARGET, truncation=True)
    inp['labels'] = tgt['input_ids']
    return inp

remove_cols = ds['train'].column_names
tokenized = ds.map(preprocess, batched=True, remove_columns=remove_cols)
tokenized

In [ ]:
import numpy as np, evaluate
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments)

rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, tok.pad_token_id)
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    decoded_preds  = tok.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tok.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels,
                           use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

collator = DataCollatorForSeq2Seq(tok, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-5,
    num_train_epochs=3,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    generation_num_beams=4,
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print('Saved fine-tuned model to', OUT_DIR)

In [ ]:
# Sanity-check generation on a held-out test example
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device).eval()
sample = ds['test'][0]
inputs = tok(sample['input_text'], max_length=MAX_INPUT, truncation=True,
             return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(**inputs, num_beams=4, min_length=80, max_length=MAX_TARGET,
                         no_repeat_ngram_size=3, length_penalty=1.0, early_stopping=True)
print('PRED:\n', tok.decode(out[0], skip_special_tokens=True))
print('\nGOLD:\n', sample['target_text'])